In [ ]:
!git clone https://github.com/kaizar-rang/yanglab-acne.git
%cd yanglab-acne
%pip install roboflow kaggle grad-cam -q
%pip install -r requirements.txt -q
print("\nDone. Restart the runtime (Runtime -> Restart session), then continue from Cell 2.")

In [ ]:
import os
import json
import getpass
from pathlib import Path

%cd /content/yanglab-acne

# Credentials
roboflow_key = getpass.getpass("Enter ROBOFLOW_API_KEY: ")
kaggle_username = getpass.getpass("Enter KAGGLE_USERNAME: ")
kaggle_key = getpass.getpass("Enter KAGGLE_KEY: ")

os.environ["ROBOFLOW_API_KEY"] = roboflow_key

# Write kaggle credentials
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
with open(kaggle_dir / "kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)

# Download ACNE04 in YOLOv5 format (for patch extraction)
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("acne-vulgaris-detection").project("acne04-detection")
version = project.version(1)
version.download("yolov5", location="data/acne04", overwrite=True)

# Download DermNet
!kaggle datasets download -d shubhamgoel27/dermnet -p data/dermnet --unzip

# Run patch extractor
!python src/data/patch_extractor.py

print("Setup complete.")

In [ ]:
import torch
import torchvision
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Verify patches exist
from pathlib import Path
patch_dir = Path("data/patches")
train_acne = len(list((patch_dir / "train" / "acne").glob("*.jpg")))
train_clear = len(list((patch_dir / "train" / "clear").glob("*.jpg")))
print(f"\nTrain patches — acne: {train_acne}, clear: {train_clear}")
print(f"Ratio: {train_acne/train_clear:.2f}:1")

# Verify DermNet exists
dermnet_test = Path("data/dermnet/test")
conditions = [d.name for d in dermnet_test.iterdir() if d.is_dir()]
print(f"\nDermNet test conditions: {len(conditions)}")
print(f"Sample: {conditions[:3]}")

In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Training transforms — augmentation to improve generalization
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation transforms — no augmentation, just normalize
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load patch datasets
train_dataset = ImageFolder("data/patches/train", transform=train_transforms)
val_dataset = ImageFolder("data/patches/val", transform=val_transforms)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Classes: {train_dataset.classes}")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def build_classifier(num_classes=2):
    # Load pretrained EfficientNet-B0 with ImageNet weights
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    
    # Replace the classification head
    # EfficientNet-B0's classifier is model.classifier[1]
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    
    return model

model = build_classifier(num_classes=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model built. Using device: {device}")
print(f"Classifier head: {model.classifier[1]}")

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

# Class weights to handle 4.45:1 imbalance
class_counts = [len(list((Path("data/patches/train/acne")).glob("*.jpg"))),
                len(list((Path("data/patches/train/clear")).glob("*.jpg")))]
total = sum(class_counts)
weights = torch.tensor([total/c for c in class_counts]).to(device)

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=weights)

# Optimizer and scheduler
optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=20)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

# Training loop
num_epochs = 20
best_val_acc = 0

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, device)
    scheduler.step()
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "outputs/checkpoints/classifier/best.pt")
    
    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f}")

print(f"\nBest val accuracy: {best_val_acc:.3f}")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from torchvision.datasets import ImageFolder
import numpy as np

# Load best model
model.load_state_dict(torch.load("outputs/checkpoints/classifier/best.pt"))
model.eval()

# DermNet test set — map acne vs non-acne
# The acne folder in DermNet is "Acne and Rosacea Photos"
dermnet_test_dir = Path("data/dermnet/test")

class ACNEBinaryDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        
        for condition_dir in Path(root_dir).iterdir():
            if not condition_dir.is_dir():
                continue
            # Label 1 if acne, 0 if not
            label = 1 if "acne" in condition_dir.name.lower() else 0
            for img_path in condition_dir.glob("*.jpg"):
                self.samples.append((img_path, label))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

from PIL import Image

dermnet_dataset = ACNEBinaryDataset("data/dermnet/test", transform=val_transforms)
dermnet_loader = DataLoader(dermnet_dataset, batch_size=32, shuffle=False, num_workers=2)

# Run inference
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in dermnet_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

# Metrics
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
auroc = roc_auc_score(all_labels, all_probs)

print("Baseline DermNet Results (no adaptation):")
print(f"  Accuracy: {acc:.3f}")
print(f"  F1:       {f1:.3f}")
print(f"  AUROC:    {auroc:.3f}")

In [ ]:
import torchvision.transforms.functional as TF
from PIL import Image
import numpy as np

def histogram_match(source, reference):
    """Match the histogram of source image to reference image."""
    source = np.array(source).astype(np.float32)
    reference = np.array(reference).astype(np.float32)
    
    matched = np.zeros_like(source)
    for channel in range(3):
        src_hist, bins = np.histogram(source[:,:,channel].flatten(), 256, [0,256])
        ref_hist, _ = np.histogram(reference[:,:,channel].flatten(), 256, [0,256])
        
        src_cdf = src_hist.cumsum() / src_hist.sum()
        ref_cdf = ref_hist.cumsum() / ref_hist.sum()
        
        lookup = np.interp(src_cdf, ref_cdf, np.arange(256))
        matched[:,:,channel] = lookup[source[:,:,channel].astype(int)]
    
    return Image.fromarray(matched.astype(np.uint8))

# Sample 20 unlabeled DermNet train images as reference for histogram matching
dermnet_train_dir = Path("data/dermnet/train")
reference_images = []
for condition_dir in list(dermnet_train_dir.iterdir())[:5]:
    if condition_dir.is_dir():
        imgs = list(condition_dir.glob("*.jpg"))[:4]
        reference_images.extend([Image.open(p).convert("RGB") for p in imgs])

reference_images = reference_images[:20]
print(f"Using {len(reference_images)} DermNet reference images for histogram matching")

# Adapted transforms — add histogram matching + stronger augmentation
class HistogramMatchTransform:
    def __init__(self, reference_images):
        self.reference_images = reference_images
    
    def __call__(self, img):
        ref = random.choice(self.reference_images)
        return histogram_match(img, ref)

import random

adapted_transforms = transforms.Compose([
    HistogramMatchTransform(reference_images),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Retrain with adapted transforms
adapted_train_dataset = ImageFolder("data/patches/train", transform=adapted_transforms)
adapted_train_loader = DataLoader(adapted_train_dataset, batch_size=32, shuffle=True, num_workers=2)

# Reset model and retrain
model_adapted = build_classifier(num_classes=2).to(device)
optimizer_adapted = Adam(model_adapted.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_adapted = CosineAnnealingLR(optimizer_adapted, T_max=20)

os.makedirs("outputs/checkpoints/classifier_adapted", exist_ok=True)
best_val_acc_adapted = 0

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model_adapted, adapted_train_loader, optimizer_adapted, criterion, device)
    val_loss, val_acc = val_epoch(model_adapted, val_loader, criterion, device)
    scheduler_adapted.step()
    
    if val_acc > best_val_acc_adapted:
        best_val_acc_adapted = val_acc
        torch.save(model_adapted.state_dict(), "outputs/checkpoints/classifier_adapted/best.pt")
    
    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f}")

print(f"\nBest adapted val accuracy: {best_val_acc_adapted:.3f}")

In [ ]:
# Load best adapted model
model_adapted.load_state_dict(torch.load("outputs/checkpoints/classifier_adapted/best.pt"))
model_adapted.eval()

# Run inference on DermNet test set
all_preds_adapted, all_labels_adapted, all_probs_adapted = [], [], []
with torch.no_grad():
    for images, labels in dermnet_loader:
        images = images.to(device)
        outputs = model_adapted(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = outputs.argmax(1)
        all_preds_adapted.extend(preds.cpu().numpy())
        all_labels_adapted.extend(labels.numpy())
        all_probs_adapted.extend(probs.cpu().numpy())

# Metrics
acc_adapted = accuracy_score(all_labels_adapted, all_preds_adapted)
f1_adapted = f1_score(all_labels_adapted, all_preds_adapted)
auroc_adapted = roc_auc_score(all_labels_adapted, all_probs_adapted)

# Compare baseline vs adapted
print("Domain Adaptation Results:")
print(f"{'Metric':<12} {'Baseline':>10} {'Adapted':>10} {'Delta':>10}")
print("-" * 44)
print(f"{'Accuracy':<12} {acc:.3f}{'':>7} {acc_adapted:.3f}{'':>7} {acc_adapted-acc:+.3f}")
print(f"{'F1':<12} {f1:.3f}{'':>7} {f1_adapted:.3f}{'':>7} {f1_adapted-f1:+.3f}")
print(f"{'AUROC':<12} {auroc:.3f}{'':>7} {auroc_adapted:.3f}{'':>7} {auroc_adapted-auroc:+.3f}")

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt
import numpy as np
import random

# Target the last convolutional layer of EfficientNet-B0
target_layers = [model_adapted.features[-1]]

# Sample 10 DermNet test images
sample_indices = random.sample(range(len(dermnet_dataset)), 10)

fig, axes = plt.subplots(2, 10, figsize=(25, 6))

with GradCAM(model=model_adapted, target_layers=target_layers) as cam:
    for i, idx in enumerate(sample_indices):
        image, label = dermnet_dataset[idx]
        
        # Prepare input tensor
        input_tensor = image.unsqueeze(0).to(device)
        
        # Get prediction
        with torch.no_grad():
            output = model_adapted(input_tensor)
            pred = output.argmax(1).item()
            prob = torch.softmax(output, dim=1)[0, pred].item()
        
        # Generate Grad-CAM
        targets = [ClassifierOutputTarget(pred)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
        
        # Convert image for display
        img_display = image.permute(1, 2, 0).cpu().numpy()
        img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min())
        
        # Overlay CAM on image
        cam_image = show_cam_on_image(img_display.astype(np.float32), grayscale_cam, use_rgb=True)
        
        # Original image
        axes[0, i].imshow(img_display)
        axes[0, i].axis('off')
        axes[0, i].set_title(f'True: {"acne" if label==1 else "clear"}', fontsize=7)
        
        # Grad-CAM overlay
        axes[1, i].imshow(cam_image)
        axes[1, i].axis('off')
        axes[1, i].set_title(f'Pred: {"acne" if pred==1 else "clear"} ({prob:.2f})', fontsize=7)

axes[0, 0].set_ylabel('Original', fontsize=9)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=9)

plt.suptitle('Grad-CAM Visualizations on DermNet Test Set (Adapted Model)', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/predictions/gradcam_dermnet.png', dpi=150)
plt.show()

In [ ]:
from google.colab import files
import os

os.makedirs("outputs/checkpoints/classifier", exist_ok=True)
os.makedirs("outputs/checkpoints/classifier_adapted", exist_ok=True)
os.makedirs("outputs/predictions", exist_ok=True)

# Zip and download everything
!zip -r classification_results.zip \
    outputs/checkpoints/classifier/best.pt \
    outputs/checkpoints/classifier_adapted/best.pt \
    outputs/predictions/gradcam_dermnet.png

files.download('classification_results.zip')
print("Download initiated. Check your browser's download folder.")